# {{PROJECT_NAME}}

**Model bakeoff** — rank N candidate LLMs × M serving modes against the project's
agent tasks using deterministic gates + a fixed LLM-as-judge.

Everything the pipeline runs lives in the cells tagged `kfp_step` / `kfp_pipeline`
below. `scripts/build_pipeline.py` (or the **Build** cell) concatenates them into
`pipeline.py` — **do not edit `pipeline.py` by hand.**

The matrix (`models`, `serving_modes`, `tasks`), the judge, gate thresholds, and
score weights are all in `config.yaml`.

## DAG

```
load_dataset ──► for each (model, mode)  [SERIAL — one GPU]:
                     serve_model ──► <harness> × tasks ──► teardown_model
                 ──► judge_and_score ──► report (MLflow + runs/RUNS.md)
```

## Adding a task harness

Copy the `example_harness` cell, rename the function + `TASK`, and edit
`run_case` / `gates`. Register it in `_HARNESSES` in the Pipeline cell and add it
to `tasks:` / `gate_thresholds:` / `score_weights:` in `config.yaml`. See
`WORKBOOK.md` for the full harness contract.


## Step development


In [ ]:
from kfp import dsl
from kfp.dsl import Input, Output, Artifact
from typing import NamedTuple


### load_dataset

Pull the frozen eval cases from MinIO
(`s3://<bucket>/datasets/<version>/{<task>.jsonl, manifest.json}`) into one
combined JSONL artifact. Produce that snapshot with `scripts/export_dataset.py`.


In [ ]:
@dsl.component(
    base_image="python:3.11-slim",
    packages_to_install=["boto3"],
)
def load_dataset(
    tasks: list,
    s3_endpoint_url: str,
    bucket: str,
    version: str,
    access_key: str,
    secret_key: str,
    cases: Output[Artifact],
):
    """Fetch s3://<bucket>/datasets/<version>/{<task>.jsonl, manifest.json}.

    Writes one combined JSONL to `cases` — one line per case:
        {"task", "case_id", "provenance", "inputs", "reference", ...}
    """
    import json, pathlib
    import boto3

    s3 = boto3.client(
        "s3",
        endpoint_url=s3_endpoint_url,
        aws_access_key_id=access_key,
        aws_secret_access_key=secret_key,
        region_name="us-east-1",
    )
    prefix = f"datasets/{version}"

    manifest = json.loads(
        s3.get_object(Bucket=bucket, Key=f"{prefix}/manifest.json")["Body"].read()
    )
    print(f"manifest: created_at={manifest.get('created_at', '?')} "
          f"source={manifest.get('source', '?')} counts={manifest.get('counts', {})}")

    out = pathlib.Path(cases.path)
    out.parent.mkdir(parents=True, exist_ok=True)
    n = 0
    with out.open("w") as fh:
        for task in tasks:
            body = s3.get_object(
                Bucket=bucket, Key=f"{prefix}/{task}.jsonl"
            )["Body"].read().decode()
            k = 0
            for line in body.splitlines():
                line = line.strip()
                if not line:
                    continue
                row = json.loads(line)
                row["task"] = task
                row.setdefault("provenance", "real")
                row.setdefault("case_id", f"{task}-{k}")
                fh.write(json.dumps(row) + "\n")
                n += 1
                k += 1
            print(f"  {task}: {k} cases")
    print(f"wrote {n} cases -> {out}")
    cases.metadata["n_cases"] = n


### serve_model

Make one candidate reachable over an OpenAI-compatible `/v1` and return its
`base_url` / `model_name` / `handle`.

- **`ollama`** — evict every other resident model on the shared Ollama, `pull`
  the target, preload it with `keep_alive: -1`. Nothing to stand up.
- **`guided`** — create a short-lived vLLM `Deployment` + `Service` (xgrammar
  guided decoding, `--quantization` from `config.yaml`; **fp8 on GB10, never
  nvfp4**), poll `/health`. Needs the pipeline ServiceAccount to have
  deployments/services RBAC in `guided_namespace` — see `manifests/bakeoff-rbac.yaml`.


In [ ]:
@dsl.component(
    base_image="python:3.11-slim",
    packages_to_install=["requests", "kubernetes"],
)
def serve_model(
    model_spec: dict,
    mode: str,
    serving_cfg: dict,
) -> NamedTuple("Served", [("base_url", str), ("model_name", str), ("handle", str)]):
    """Bring up `model_spec` under serving `mode`. See the cell markdown for modes."""
    import time
    from collections import namedtuple
    import requests

    Served = namedtuple("Served", ["base_url", "model_name", "handle"])

    if mode == "ollama":
        root = serving_cfg["ollama_base_url"].rstrip("/")
        api = root[:-3].rstrip("/") if root.endswith("/v1") else root
        tag = model_spec["ollama_tag"]

        try:
            ps = requests.get(f"{api}/api/ps", timeout=30).json()
            for m in ps.get("models", []):
                if m.get("name") and m["name"] != tag:
                    requests.post(f"{api}/api/generate",
                                  json={"model": m["name"], "keep_alive": 0}, timeout=60)
                    print(f"evicted {m['name']}")
        except Exception as e:
            print(f"eviction skipped: {e}")

        pull = requests.post(f"{api}/api/pull", json={"model": tag}, stream=True,
                             timeout=serving_cfg.get("ollama_pull_timeout_s", 1800))
        for _ in pull.iter_lines():
            pass
        pull.raise_for_status()

        requests.post(f"{api}/api/generate",
                      json={"model": tag, "prompt": "", "keep_alive": -1}, timeout=600)
        print(f"preloaded {tag}")
        return Served(base_url=f"{api}/v1", model_name=tag, handle=tag)

    if mode == "guided":
        from kubernetes import client as kc, config as kconf

        kconf.load_incluster_config()
        apps, core = kc.AppsV1Api(), kc.CoreV1Api()
        ns = serving_cfg.get("guided_namespace", "kubeflow")
        name = "bakeoff-vllm"
        hf_id = model_spec["hf_id"]
        quant = model_spec.get("quant", "fp8")

        for delete in (lambda: apps.delete_namespaced_deployment(name, ns),
                       lambda: core.delete_namespaced_service(name, ns)):
            try:
                delete()
            except Exception:
                pass
        time.sleep(5)

        container = kc.V1Container(
            name="vllm",
            image=serving_cfg.get("vllm_image", "vllm/vllm-openai:latest"),
            args=["--model", hf_id, "--quantization", quant,
                  "--guided-decoding-backend", "xgrammar",
                  "--max-model-len", str(serving_cfg.get("vllm_max_model_len", 8192))],
            ports=[kc.V1ContainerPort(container_port=8000)],
            env=[kc.V1EnvVar(
                name="HF_TOKEN",
                value_from=kc.V1EnvVarSource(secret_key_ref=kc.V1SecretKeySelector(
                    name="mlabs-api-keys", key="HF_TOKEN", optional=True)))],
            resources=kc.V1ResourceRequirements(limits={"nvidia.com/gpu": "1"}),
            volume_mounts=[kc.V1VolumeMount(
                name="hf", mount_path="/root/.cache/huggingface")],
            readiness_probe=kc.V1Probe(
                http_get=kc.V1HTTPGetAction(path="/health", port=8000),
                initial_delay_seconds=30, period_seconds=15, failure_threshold=240),
        )
        pod = kc.V1PodSpec(
            containers=[container],
            volumes=[kc.V1Volume(
                name="hf",
                persistent_volume_claim=kc.V1PersistentVolumeClaimVolumeSource(
                    claim_name="hf-model-cache"))],
        )
        apps.create_namespaced_deployment(ns, kc.V1Deployment(
            metadata=kc.V1ObjectMeta(name=name, namespace=ns, labels={"app": name}),
            spec=kc.V1DeploymentSpec(
                replicas=1,
                selector=kc.V1LabelSelector(match_labels={"app": name}),
                template=kc.V1PodTemplateSpec(
                    metadata=kc.V1ObjectMeta(labels={"app": name}), spec=pod))))
        core.create_namespaced_service(ns, kc.V1Service(
            metadata=kc.V1ObjectMeta(name=name, namespace=ns),
            spec=kc.V1ServiceSpec(
                selector={"app": name},
                ports=[kc.V1ServicePort(port=8000, target_port=8000)])))

        host = f"{name}.{ns}.svc.cluster.local:8000"
        deadline = time.time() + serving_cfg.get("guided_readiness_timeout_s", 900)
        while time.time() < deadline:
            try:
                if requests.get(f"http://{host}/health", timeout=10).ok:
                    print(f"vLLM ready at {host}")
                    return Served(base_url=f"http://{host}/v1",
                                  model_name=hf_id, handle=f"{ns}/{name}")
            except Exception:
                pass
            time.sleep(15)
        raise TimeoutError(f"vLLM /health not ready within "
                           f"{serving_cfg.get('guided_readiness_timeout_s', 900)}s")

    raise ValueError(f"unknown serving mode: {mode!r}")


### teardown_model

Undo `serve_model`: unload the Ollama model (`keep_alive: 0`) or delete the
transient vLLM `Deployment` + `Service`. Runs after every harness for the combo.


In [ ]:
@dsl.component(
    base_image="python:3.11-slim",
    packages_to_install=["requests", "kubernetes"],
)
def teardown_model(mode: str, handle: str, serving_cfg: dict):
    """Release the GPU held by the combo `serve_model` set up."""
    import requests

    if mode == "ollama":
        root = serving_cfg["ollama_base_url"].rstrip("/")
        api = root[:-3].rstrip("/") if root.endswith("/v1") else root
        try:
            requests.post(f"{api}/api/generate",
                          json={"model": handle, "keep_alive": 0}, timeout=60)
            print(f"unloaded {handle}")
        except Exception as e:
            print(f"unload failed (non-fatal): {e}")
        return

    if mode == "guided":
        from kubernetes import client as kc, config as kconf

        kconf.load_incluster_config()
        ns, name = handle.split("/", 1)
        for delete in (lambda: kc.AppsV1Api().delete_namespaced_deployment(name, ns),
                       lambda: kc.CoreV1Api().delete_namespaced_service(name, ns)):
            try:
                delete()
            except Exception as e:
                print(f"delete failed (non-fatal): {e}")
        print(f"deleted {handle}")
        return

    print(f"teardown: nothing to do for mode {mode!r}")


### example_harness  ← copy this to add a task

**Harness contract.** Every task harness is a `kfp_step` component with *exactly*
this signature. It:

1. reads `cases` (the JSONL from `load_dataset`), keeps rows where
   `row["task"] == TASK`;
2. `run_case(case)` → the model's structured output as a dict, bounded by
   `case_timeout_s` (every model call MUST be time-bounded — GB10 silent-hang lesson);
3. `gates(case, output)` → `{gate_name: bool}` — deterministic, no model;
4. writes one JSONL row per case to
   `<runs_dir>/<run_id>/<model>__<mode>__<TASK>.jsonl`:
   `{model, mode, task, case_id, provenance, gate_pass, gates, output, reference, error}`.

The judge runs later in `judge_and_score`, so emit `output` and `reference`
verbatim — don't score here.


In [ ]:
@dsl.component(
    base_image="python:3.11-slim",
    packages_to_install=["openai>=1.0"],
)
def example_harness(
    cases: Input[Artifact],
    model_id: str,
    model_name: str,
    base_url: str,
    mode: str,
    run_id: str,
    gate_cfg: dict,
    case_timeout_s: int,
    runs_dir: str,
):
    """Reference harness: prompt the model for a JSON object, gate on key count.

    Copy this cell, rename the function and TASK, and rewrite run_case / gates.
    """
    import json, pathlib
    from openai import OpenAI

    TASK = "example_harness"
    client = OpenAI(base_url=base_url, api_key="x", timeout=case_timeout_s)

    def run_case(case: dict) -> dict:
        raw = case.get("inputs")
        prompt = raw.get("prompt") if isinstance(raw, dict) else str(raw)
        resp = client.chat.completions.create(
            model=model_name,
            messages=[
                {"role": "system", "content": "Reply with a single JSON object. No prose."},
                {"role": "user", "content": prompt},
            ],
            response_format={"type": "json_object"},
            temperature=0,
        )
        return json.loads(resp.choices[0].message.content or "{}")

    def gates(case: dict, output) -> dict:
        is_obj = isinstance(output, dict)
        return {
            "is_object": is_obj,
            "min_output_keys": is_obj and len(output) >= int(gate_cfg.get("min_output_keys", 1)),
        }

    src = [
        json.loads(l)
        for l in pathlib.Path(cases.path).read_text().splitlines()
        if l.strip()
    ]
    mine = [c for c in src if c.get("task") == TASK]
    print(f"{TASK}: {len(mine)} cases  ({model_id} / {mode})")

    slug = model_id.replace("/", "-").replace(":", "-")
    out = pathlib.Path(runs_dir) / run_id / f"{slug}__{mode}__{TASK}.jsonl"
    out.parent.mkdir(parents=True, exist_ok=True)

    with out.open("w") as fh:
        for case in mine:
            row = {
                "model": model_id, "mode": mode, "task": TASK,
                "case_id": str(case.get("case_id", "?")),
                "provenance": case.get("provenance", "real"),
                "reference": case.get("reference"),
                "output": None, "gates": {}, "gate_pass": False, "error": None,
            }
            try:
                output = run_case(case)
                g = gates(case, output)
                row.update(output=output, gates=g, gate_pass=all(g.values()))
            except Exception as e:
                row["error"] = f"{type(e).__name__}: {e}"
            fh.write(json.dumps(row, default=str) + "\n")
    print(f"wrote {out}")


### judge_and_score

Judge every harness row on the fixed 1–5 rubric, then compute the weighted
composite leaderboard. The pure helpers are spliced in from `evallib/` at build
time (`# inline:` — they are also unit-tested under `tests/`).


In [ ]:
@dsl.component(
    base_image="python:3.11-slim",
    packages_to_install=["openai>=1.0"],
)
def judge_and_score(
    runs_dir: str,
    run_id: str,
    judge_model: str,
    judge_base_url: str,
    tasks: list,
    score_weights: dict,
    case_timeout_s: int,
    leaderboard: Output[Artifact],
):
    """LLM-judge every row, then rank (model, mode) by weighted composite."""
    import glob, json, pathlib
    from openai import OpenAI

    # inline: evallib/rubric.py

    # inline: evallib/scoring.py

    judge = OpenAI(base_url=judge_base_url, api_key="x", timeout=case_timeout_s)

    def judge_row(row: dict):
        if row.get("error") or row.get("output") is None:
            return None
        task_prompt = (
            f"TASK: {row['task']}\n"
            f"REFERENCE (may be null): {json.dumps(row.get('reference'), default=str)}\n"
            f"CANDIDATE OUTPUT: {json.dumps(row.get('output'), default=str)}\n"
            "Score the candidate 1-5 for correctness and grounding in the reference."
        )
        try:
            resp = judge.chat.completions.create(
                model=judge_model,
                messages=wrap_judge_prompt(task_prompt),
                response_format={"type": "json_object"},
                temperature=0,
            )
            return parse_judge_response(resp.choices[0].message.content or "")["score"]
        except Exception as e:
            print(f"judge error on {row.get('case_id')}: {e}")
            return None

    rows = []
    for fp in sorted(glob.glob(f"{runs_dir}/{run_id}/*.jsonl")):
        for line in pathlib.Path(fp).read_text().splitlines():
            if not line.strip():
                continue
            row = json.loads(line)
            row["judge_score"] = judge_row(row)
            rows.append(row)
    print(f"judged {len(rows)} rows from {runs_dir}/{run_id}")

    board = score_matrix(rows, {"tasks": tasks, "score_weights": score_weights})
    md_table = leaderboard_markdown(board)
    print(md_table)

    out = pathlib.Path(leaderboard.path)
    out.parent.mkdir(parents=True, exist_ok=True)
    out.write_text(json.dumps(
        {"leaderboard": board, "markdown": md_table, "n_rows": len(rows)},
        indent=2, default=str))


### report

Log the leaderboard to MLflow (params + per-combo composite metrics + the
Markdown/JSON artifacts) and append a dated winner block to
`<runs_dir>/RUNS.md`.


In [ ]:
@dsl.component(
    base_image="python:3.11-slim",
    packages_to_install=["mlflow"],
)
def report(
    leaderboard: Input[Artifact],
    run_id: str,
    runs_dir: str,
    mlflow_tracking_uri: str,
    mlflow_experiment_name: str,
    project_name: str,
):
    """Persist the bakeoff result to MLflow + RUNS.md."""
    import datetime, json, pathlib
    import mlflow

    data = json.loads(pathlib.Path(leaderboard.path).read_text())
    board = data["leaderboard"]
    md_table = data["markdown"]
    if not board:
        raise RuntimeError("empty leaderboard — no rows scored")
    winner = board[0]

    mlflow.set_tracking_uri(mlflow_tracking_uri)
    mlflow.set_experiment(mlflow_experiment_name)
    with mlflow.start_run(run_name=f"{run_id}-bakeoff"):
        mlflow.log_param("project", project_name)
        mlflow.log_param("n_combos", len(board))
        mlflow.log_param("n_rows", data.get("n_rows"))
        mlflow.log_param("winner_model", winner["model"])
        mlflow.log_param("winner_mode", winner["mode"])
        mlflow.log_metric("winner_composite", winner["composite"])
        for e in board:
            mlflow.log_metric(f"composite::{e['model']}::{e['mode']}", e["composite"])
        mlflow.log_text(md_table, "leaderboard.md")
        mlflow.log_dict(data, "leaderboard.json")

    stamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
    block = (
        f"\n## {run_id} — {stamp}\n\n"
        f"**Winner: `{winner['model']}` ({winner['mode']}) — "
        f"composite {winner['composite']:.3f}**\n\n"
        f"{md_table}\n"
    )
    runs_md = pathlib.Path(runs_dir) / "RUNS.md"
    runs_md.parent.mkdir(parents=True, exist_ok=True)
    with runs_md.open("a") as fh:
        fh.write(block)
    print(block)


### Pipeline

Reads `config.yaml` at build time and unrolls the `models × serving_modes`
matrix into a serial chain (one GPU). Add a harness → register it in
`_HARNESSES`.


In [ ]:
import yaml as _yaml, pathlib as _pathlib

_cfg = _yaml.safe_load(_pathlib.Path("config.yaml").read_text())
_pipeline_name = "{{PROJECT_NAME}}"
_MODELS = _cfg["models"]
_MODES = _cfg["serving_modes"]
_TASKS = _cfg["tasks"]

from kfp import kubernetes as k8s_ext

# task name -> harness component. Add a row for every harness cell you create.
_HARNESSES = {
    "example_harness": example_harness,
}
_missing = [t for t in _TASKS if t not in _HARNESSES]
if _missing:
    raise KeyError(f"config.yaml tasks not in _HARNESSES: {_missing}")


@dsl.pipeline(name="{{PROJECT_NAME}}")
def pipeline(
    run_id: str = "run-001",
    mlflow_tracking_uri: str = "http://mlflow-tracking.mlflow-system.svc.cluster.local",
    mlflow_experiment_name: str = "{{PROJECT_NAME}}",
    s3_endpoint_url: str = _cfg["dataset"]["s3_endpoint_url"],
    bucket: str = _cfg["dataset"]["bucket"],
    dataset_version: str = _cfg["dataset"]["version"],
    s3_access_key: str = _cfg["dataset"]["access_key"],
    s3_secret_key: str = _cfg["dataset"]["secret_key"],
    judge_model: str = _cfg["judge"]["model"],
    judge_base_url: str = _cfg["judge"]["base_url"],
    case_timeout_s: int = _cfg["case_timeout_s"],
):
    _RUNS = "/root/.cache/huggingface/bakeoff-runs"
    _serving = _cfg["serving"]
    _gates = _cfg.get("gate_thresholds", {})
    _weights = _cfg.get("score_weights", {})

    load = load_dataset(
        tasks=_TASKS,
        s3_endpoint_url=s3_endpoint_url,
        bucket=bucket,
        version=dataset_version,
        access_key=s3_access_key,
        secret_key=s3_secret_key,
    )

    ops = [load]
    prev_teardown = None
    for _m in _MODELS:
        for _mode in _MODES:
            serve = serve_model(model_spec=_m, mode=_mode, serving_cfg=_serving)
            serve.after(load)
            if prev_teardown is not None:
                serve.after(prev_teardown)          # serialize combos — one GPU

            harness_ops = []
            for _t in _TASKS:
                op = _HARNESSES[_t](
                    cases=load.outputs["cases"],
                    model_id=_m["id"],
                    model_name=serve.outputs["model_name"],
                    base_url=serve.outputs["base_url"],
                    mode=_mode,
                    run_id=run_id,
                    gate_cfg=_gates.get(_t, {}),
                    case_timeout_s=case_timeout_s,
                    runs_dir=_RUNS,
                )
                op.after(serve)
                harness_ops.append(op)

            td = teardown_model(
                mode=_mode, handle=serve.outputs["handle"], serving_cfg=_serving,
            )
            for op in harness_ops:
                td.after(op)
            prev_teardown = td
            ops += [serve, *harness_ops, td]

    js = judge_and_score(
        runs_dir=_RUNS,
        run_id=run_id,
        judge_model=judge_model,
        judge_base_url=judge_base_url,
        tasks=_TASKS,
        score_weights=_weights,
        case_timeout_s=case_timeout_s,
    )
    if prev_teardown is not None:
        js.after(prev_teardown)

    rep = report(
        leaderboard=js.outputs["leaderboard"],
        run_id=run_id,
        runs_dir=_RUNS,
        mlflow_tracking_uri=mlflow_tracking_uri,
        mlflow_experiment_name=mlflow_experiment_name,
        project_name=_pipeline_name,
    )
    rep.after(js)

    _SECRET = "mlabs-api-keys"
    _SECRET_KEYS = {
        "OPENAI_API_KEY": "OPENAI_API_KEY",
        "HF_TOKEN": "HF_TOKEN",
        "LANGCHAIN_API_KEY": "LANGCHAIN_API_KEY",
    }
    _PVC = "hf-model-cache"
    for _op in ops + [js, rep]:
        k8s_ext.mount_pvc(_op, pvc_name=_PVC, mount_path="/root/.cache/huggingface")
        k8s_ext.use_secret_as_env(_op, _SECRET, _SECRET_KEYS)


## Build → `pipeline.py`


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "scripts/build_pipeline.py"], check=True)


## Compile & Submit


In [ ]:
from kfp import compiler
from pipeline import pipeline

compiler.Compiler().compile(pipeline_func=pipeline, package_path="/tmp/pipeline.yaml")
print("Compiled: /tmp/pipeline.yaml")


In [ ]:
# Requires SSH tunnel: ssh -L 8080:localhost:8080 <user>@spark-79b7.local
import kfp

client = kfp.Client(host="http://localhost:8080")


In [ ]:
run = client.create_run_from_pipeline_package(
    pipeline_file="/tmp/pipeline.yaml",
    arguments={"run_id": "notebook-run"},
    run_name="notebook-run",
)
print(f"Run ID: {run.run_id}")


In [ ]:
import time

run_id = run.run_id  # or paste a run ID here
while True:
    r = client.get_run(run_id)
    print(r.state)
    if r.state in ("SUCCEEDED", "FAILED", "CANCELED", "SKIPPED"):
        break
    time.sleep(30)
